In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from joblib import load

In [5]:
def apply_anchors(X, y_test, predictor_fn, feature_names, num_samples=10000, threshold=1.0):
    """
    Applies Anchor explanations on a test dataset and extracts only 100% precision rules.

    :param X: DataFrame containing test samples.
    :param y_test: Ground truth labels.
    :param predictor_fn: Function that returns model predictions.
    :param feature_names: List of feature names.
    :param num_samples: Number of samples for Anchors (higher gives better rules but is slower).
    :param threshold: Minimum precision required (default: 1.0 for 100% precision).
    :return: DataFrame with extracted rules that meet the precision threshold.
    """
    explainer = AnchorTabular(predictor_fn, feature_names)
    explainer.fit(X.to_numpy())

    rules_list = []

    for idx in tqdm(range(len(X))):
        instance = X.iloc[idx].to_numpy()
        exp = explainer.explain(instance, threshold=threshold, batch_size=50, num_samples=num_samples)

        if exp.precision() == 1.0:  # Keep only rules with 100% precision
            rules_list.append({
                "Instance Index": idx,
                "Anchor": " AND ".join(exp.names()),
                "Precision": exp.precision(),
                "Coverage": exp.coverage(),
                "Prediction": exp.predictor()
            })

    return pd.DataFrame(rules_list)

In [3]:
import numpy as np
import pandas as pd
from joblib import load
from tqdm import tqdm
from aix360.algorithms.rbm import BooleanRuleCG
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Load Data
X_train = pd.read_csv('../Datasets/KDD99/X_train.csv')
y_train = pd.read_csv('../Datasets/KDD99/y_train.csv')

# Ensure X_train is binarized (convert categorical to dummies)
X_train = pd.get_dummies(X_train, drop_first=True)

# Ensure y_train is a 1D NumPy array
y_train = np.ravel(y_train)

# Train Boolean Rule Model
br = BooleanRuleCG()
br.fit(X_train, y_train)

# Try again
print(br.explain())


Learning DNF rule with complexity parameters lambda0=0.001, lambda1=0.001


SolverError: Solver 'ECOS' failed. Try another solver, or solve with verbose=True for more information.